In [2]:
import pdb
import sqlite3
import logging
import datetime as dt

In [3]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [4]:
def dict_factory(cursor, row):
    fields = [column[0] for column in cursor.description]
    return {key: value for key, value in zip(fields, row)}

In [5]:
def create_dim_time(conn: sqlite3.Connection):
    sql = """
    create table if not exists dim_time (
        id integer primary key, -- automatically auto increments
        dt datetime unique,
        year int,
        month int,
        day int,
        hour int,
        minute int,
        quarter int,
        day_name text,
        day_name_abbr text,
        month_name text,
        month_name_abbr text,
        is_weekend int,
        day_period text
    )
    """
    cur = conn.cursor()
    cur.execute(sql)

In [15]:
def create_dim_product(conn: sqlite3.Connection):
    sql = """
    -- Representa: Products, Categories e Suppliers
    -- Três opções de gerenciamento dos valores:
    -- primeira inserção é a que fica
    -- última inserção é a que fica (update)
    -- guardamos todos os estados (surrogate obrigatória)
    -- (product_id, product_unit_price) => histórico de variação
    -- dos preços do produto
    create table if not exists dim_product (
        id integer primary key, -- automatically auto increments
        product_id int unique,  -- chave natural
        product_name text,
        category_name text,
        category_description text,
        supplier_company text,
        supplier_city text,
        supplier_region text,
        supplier_country text,
        product_unit_price numeric,
        product_quantity_per_unit numeric,
        product_discontinued boolean
    )
    """
    cur = conn.cursor()
    cur.execute(sql)

In [13]:
def create_dim_customer(conn: sqlite3.Connection):
    sql = """
    create table if not exists dim_customer (
        id integer primary key, -- automatically auto increments
        customer_id text,       -- chave natural
        company_name text,
        contact_name text,
        city text,
        region text,
        country text
    )
    """
    cur = conn.cursor()
    cur.execute(sql)

In [14]:
def create_dim_employee(conn: sqlite3.Connection):
    sql = """
    create table if not exists dim_employee (
        id integer primary key, -- automatically auto increments
        employee_id int unique, -- chave natural
        full_name text,
        title text,
        birth_date date,
        hire_date date,
        city text,
        region text,
        country text,
        reports_to_name text
    )
    """
    cur = conn.cursor()
    cur.execute(sql)

In [9]:
def create_dim_shipper(conn: sqlite3.Connection):
    sql = """
    create table dim_shipper (
        id integer primary key, -- automatically auto increments
        shipper_id int unique,  -- chave natural
        company_name text,
        phone text
    )
    """
    cur = conn.cursor()
    cur.execute(sql)

In [10]:
def create_dim_ship_location(conn: sqlite3.Connection):
    sql = """
    create table dim_ship_location (
        id integer primary key, -- automatically auto increments
        city text,
        region text,
        country text,
        unique (city, region, country)
    )
    """
    cur = conn.cursor()
    cur.execute(sql)

In [11]:
def create_fact_sales(conn: sqlite3.Connection):
    sql = """
    -- Representa: Orders e Order Details
    create table fact_sales (
        id integer primary key, -- automatically auto increments
        order_id int unique,    -- chave natural
        -- Chaves Estrangeiras
        time_id int references dim_time(id),
        customer_id int references dim_customer(id),
        product_id int references dim_product(id),
        employee_id int references dim_employee(id),
        shipper_id int references dim_shipper(id),
        ship_location_id int references dim_ship_location(id),
        -- Metricas
        unit_price_sold numeric,
        quantity int,
        discount numeric,
        revenue_net numeric, -- (Price * Qty) * (1 - Discount)
        freight_apportioned numeric -- granularidade do campo diferente da granularidade da dimensão
    )
    """

In [11]:
def insert_dim_time(
    src_conn: sqlite3.Connection,
    dst_conn: sqlite3.Connection,
    dt_range: tuple[dt.datetime, dt.datetime] = None
):
    def fetch_start_end_dates():
        sql_dates = """
        select min(OrderDate) as dt_start,
               max(OrderDate) as dt_end
        from Orders
        """
        cur = src_conn.cursor()
        res = cur.execute(sql_dates).fetchone()
        return res["dt_start"], res["dt_end"]

    def calculate_day_period(dt: dt.datetime):
        val = int(f"{dt.hour}{dt.minute}")
        if 600 <= val and val <= 1159:
            return "Morning"
        elif 1200 <= val and val <= 1759:
            return "Afternoon"
        elif 1800 <= val and val <= 2359:
            return "Night"
        else:
            return "Midnight"

    if not dt_range:
        dt_start, dt_end = fetch_start_end_dates()
        dt_start = dt.datetime.strptime(dt_start, "%Y-%m-%d %H:%M:%S")
        dt_end = dt.datetime.strptime(dt_end, "%Y-%m-%d %H:%M:%S")
    else:
        dt_start, dt_end = dt_range

    dt_aux = dt_start
    values = []
    while dt_aux <= dt_end:
        values.append(
            {
                "dt": dt_aux.strftime('%Y-%m-%d %H:%M:%S'),
                "year": dt_aux.year,
                "month": dt_aux.month,
                "day": dt_aux.day,
                "hour": dt_aux.hour,
                "minute": dt_aux.minute,
                "quarter": (dt_aux.month - 1) // 3 + 1,
                "day_name": dt_aux.strftime("%A"),
                "day_name_abbr": dt_aux.strftime("%a"),
                "month_name": dt_aux.strftime("%B"),
                "month_name_abbr": dt_aux.strftime("%b"),
                "is_weekend": dt_aux.weekday() > 4,
                "day_period": calculate_day_period(dt_aux),
            }
        )
        if len(values) % 50_000 == 0:
            logger.info(f"Inserted {len(values)} date times")
            cur = dst_conn.cursor()
            cur.executemany("""
                insert into dim_time (
                    dt, year, month, day, hour, minute, quarter, day_name, day_name_abbr, month_name, month_name_abbr, is_weekend, day_period
                ) values (
                    :dt, :year, :month, :day, :hour, :minute, :quarter, :day_name, :day_name_abbr, :month_name, :month_name_abbr, :is_weekend, :day_period
                )
                """,
                values
            )
            dst_conn.commit()
            cur.close()
            values = []
        dt_aux += dt.timedelta(minutes=1)

In [16]:
def insert_dim_product(
    src_conn: sqlite3.Connection,
    dst_conn: sqlite3.Connection,
):
    def fetch_joined_info():
        sql = """
        select
            p.ProductID as product_id,
            p.ProductName as product_name,
            c.CategoryName as category_name,
            c.Description as category_description,
            s.CompanyName as supplier_company,
            s.City as supplier_city,
            s.Region as supplier_region,
            s.Country as supplier_country,
            p.UnitPrice as product_unit_price,
            p.QuantityPerUnit as product_quantity_per_unit,
            case
                when p.Discontinued = '0'
                    then false
                when p.Discontinued = '1'
                    then true
                else false
            end as product_discontinued
        from Products p
        inner join Categories c on p.CategoryID = c.CategoryID
        inner join Suppliers s on p.SupplierID = s.SupplierID
        """
        cur = src_conn.cursor()
        res = cur.execute(sql).fetchall()
        return res

    values = fetch_joined_info()
    cur = dst_conn.cursor()
    cur.executemany("""
        insert into dim_product (
            product_id, product_name, category_name, category_description, supplier_company, supplier_city, supplier_region, supplier_country, product_unit_price,
            product_quantity_per_unit, product_discontinued
        ) values (
            :product_id, :product_name, :category_name, :category_description, :supplier_company, :supplier_city, :supplier_region, :supplier_country, :product_unit_price,
            :product_quantity_per_unit, :product_discontinued
        )
        """,
        values
    )
    dst_conn.commit()
    cur.close()

In [17]:
with sqlite3.connect("northwind.db") as oltp_conn, sqlite3.connect("northwind-dw.db") as olap_conn:
    oltp_conn.row_factory = dict_factory
    olap_conn.row_factory = dict_factory
    # create_dim_time(olap_conn)
    # create_dim_product(olap_conn)
    # create_dim_customer(olap_conn)
    # create_dim_employee(olap_conn)
    # create_dim_shipper(olap_conn)
    # create_dim_ship_location(olap_conn)
    # insert_dim_time(oltp_conn, olap_conn)
    # insert_dim_product(oltp_conn, olap_conn)